# Import

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict, Features, Value, Sequence
import huggingface_hub
import json
import os

from PIL import Image, ImageDraw
from tqdm import tqdm 

import sys 
sys.path.append("../../decap/src")

from bbox_utils import draw_bounding_boxes

# Utils

In [ ]:
def get_correct_path(data_dir, coco_id, splits=['train2014', 'val2014', 'test2014']):
    """
    Given a COCO image ID (without leading zeros), detect which split it belongs to
    and return the split name, full path, and filename.
    """
    # Zero-pad COCO ID to 12 digits
    coco_id_str = str(coco_id).zfill(12)

    for split in splits:
        filename = f"COCO_{split}_{coco_id_str}.jpg"
        potential_path = os.path.join(data_dir, split, filename)

        if os.path.exists(potential_path):
            return split, potential_path, filename

    raise FileNotFoundError(
        f"Image with ID {coco_id} not found in splits: {splits}"
    )


def preprocess(img, ann, preprocess_type='crop'):
    if preprocess_type == 'crop':
        cropped_img = img.crop((ann[0], ann[1], ann[2], ann[3]))
        return cropped_img
    
    elif preprocess_type == 'visual_prompting':
        return draw_bounding_boxes(img, [ann])

In [ ]:
v = '../../image_data.json'
with open(v, 'r') as f:
    metadata = json.load(f)

vg2coco = {x['image_id']: x['coco_id'] for x in metadata if x['coco_id'] is not None}

# Loading

In [ ]:
dataset_path = '/raid/datasets/densecaptioning-annotations/data/vg/controlcap/vgcoco/test.json'
# Load the JSON file
with open(dataset_path, 'r') as f:
    data = json.load(f)

# Creating Pandas

In [ ]:
base_path = '/raid/datasets/MLLM_patchioner'
coco_path = '../../../coco'
taskdir = os.path.join(base_path, 'dense')
cropdir = os.path.join(taskdir, 'crops')
visualprompt_dir = os.path.join(taskdir, 'visual_prompts')
os.makedirs(base_path, exist_ok=True)
os.makedirs(taskdir, exist_ok=True)
os.makedirs(cropdir, exist_ok=True)
os.makedirs(visualprompt_dir, exist_ok=True)

sentences = []
img_ids = []
urls_crop = []
urls_visualprompt = []
filenames = []
splits = []
sent_ids = []
filepaths = []
for ann in tqdm(data['annotations'], total=len(data['annotations'])):
    img_id = vg2coco[ann['image_id']]
    caption = ann['caption']
    bbox = ann['bbox']
    idx = ann['id']
    img_ids.append(img_id)
    sentences.append([caption])
    urls_crop.append(os.path.join(cropdir, f"{img_id}_{idx}.jpg"))
    urls_visualprompt.append(os.path.join(visualprompt_dir, f"{img_id}_{idx}.jpg"))
    split, current_img_path, filename = get_correct_path(coco_path, img_id)
    filepath = split
    split = 'test' # split[:-4]
    splits.append(split)
    filenames.append(f"COCO_val2014_{str(idx).zfill(12)}.jpg")# filenames.append(filename)
    sent_ids.append([len(sent_ids)])
    filepaths.append(filepath)

    # Preprocess and save the image
    # output_img_path_crop = os.path.join(cropdir, f"{img_id}_{idx}.jpg")
    # img = Image.open(current_img_path)
    # cropped_img = preprocess(img, bbox, preprocess_type='crop')
    # if 0 in cropped_img.size:
    #     invalid_traces += 1
    #     img.save(output_img_path_crop)  # Save the original image if crop is invalid
    # else:
    #     cropped_img.save(output_img_path_crop)
    # output_img_path_vp = os.path.join(visualprompt_dir, f"{img_id}_{idx}.jpg")
    # visual_prompt_img = preprocess(img, bbox, preprocess_type='visual_prompting')
    # visual_prompt_img.save(output_img_path_vp)


len(sentences), len(img_ids), len(urls_crop), len(urls_visualprompt), len(filenames), len(splits), len(sent_ids), len(filepaths)

In [ ]:
# Create a Pandas DataFrame
df_crop = pd.DataFrame({
    'filepath': filepaths,
    'sentids': sent_ids,
    'filename': filenames,
    'imgid': img_ids,
    'split': splits,
    'sentences': sentences,
    'cocoid': img_ids,
    'url': urls_crop,
})
df_crop

In [ ]:
df_visualprompt = pd.DataFrame({
    'filepath': filepaths,
    'sentids': sent_ids,
    'filename': filenames,
    'imgid': img_ids,
    'split': splits,
    'sentences': sentences,
    'cocoid': img_ids,
    'url': urls_visualprompt,
})
df_visualprompt['url']

# Pushing to the hub

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict, Features, Value, Sequence
import huggingface_hub


# -----------------------------------------------------
# 1. Load your pandas DataFrame
# -----------------------------------------------------
dfs = [df_crop, df_visualprompt]  # or df_visualprompt
repo_ids = ["user/coco-karpathy_dense_crops", "user/coco-karpathy_dense_visualprompt"]
for df, repo_id in zip(dfs, repo_ids):
    # Expected DataFrame columns:
    # filepath : str
    # sentids  : list[int]
    # filename : str
    # imgid    : int
    # split    : str ("train" / "val" / "test")
    # sentences: list[str]
    # cocoid   : int
    # url      : str
    print(f"\nProcessing dataset for repo_id: {repo_id}\n")


    # -----------------------------------------------------
    # 2. Define the dataset schema
    # -----------------------------------------------------
    features = Features({
        "filepath": Value("string"), #    'filepath': filepaths,
        "sentids": Sequence(Value("int64")), #     'sentids': sent_ids,
        "filename": Value("string"), #     'filename': filenames,
        "imgid": Value("int64"), #     'imgid': img_ids,
        "split": Value("string"), #     'split': splits,
        "sentences": Sequence(Value("string")), #     'sentences': sentences,
        "cocoid": Value("int64"), #     'cocoid': img_ids,
        "url": Value("string") #     'url': urls_crop,
    })


    # -----------------------------------------------------
    # 3. Create Dataset or DatasetDict depending on splits
    # -----------------------------------------------------
    if "split" in df.columns:
        print("Detected split column → building DatasetDict")

        dsets = {}
        for split_name in df["split"].unique():
            split_df = df[df["split"] == split_name]
            dsets[split_name] = Dataset.from_pandas(
                split_df,
                features=features,
                preserve_index=False
            )

        ds = DatasetDict(dsets)

    else:
        print("No split column found → building single Dataset")

        ds = Dataset.from_pandas(
            df,
            features=features,
            preserve_index=False
        )


    # -----------------------------------------------------
    # 4. Push to Hugging Face Hub
    # -----------------------------------------------------

    ds.push_to_hub(
        repo_id,
        private=True,
        commit_message="Initial upload of modified COCO-Karpathy dataset"
    )

    print(f"\nDataset successfully pushed to https://huggingface.co/{repo_id}\n")